# Import libraries

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pickle as pkl

from lib.helpers import sample_data, pad_zeros
from lib.algorithms import fwht_batch

# Load sample data

In [2]:
df = sample_data("../data/processed/arxiv_embeddings.parquet")

In [3]:
df.head()

,id,content,embedding
0,0704.0001,Title: Calculation of prompt diphoton producti...,"[-0.15867826, -0.010544391, -0.0027284771, 0.0..."
1,0704.0105,Title: Rigid subsets of symplectic manifolds |...,"[-0.01407553, 0.006501307, 0.0384129, -0.03039..."
2,0704.0208,Title: Some non-braided fusion categories of r...,"[-0.045916885, -0.035438374, 0.0035416735, -0...."
3,0704.0310,Title: VLBI observations of nineteen GHz-Peake...,"[-0.043781985, 0.040595774, -0.03473763, 0.003..."
4,0704.0412,Title: Unit groups of integral finite group ri...,"[0.011277134, 0.023577133, -0.042229224, 0.074..."


# Normalise raw embeddings

In [4]:
embeddings = np.vstack(df["embedding"].values)
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)

In [5]:
embeddings.shape

(30057, 384)

# Rotate Embeddings

In [6]:
rot_embeddings, d_signs = fwht_batch(embeddings, seed=42, sign_flip=True)

In [7]:
rot_embeddings.shape

(30057, 512)

In [8]:
rot_embeddings

array([[-0.01048563, -0.0197812 ,  0.02305281, ...,  0.00577685,
         0.00428093, -0.00941336],
       [-0.04989729, -0.02426882, -0.00056059, ...,  0.04910655,
         0.04135597, -0.01228258],
       [-0.10157807,  0.02300107, -0.01774016, ...,  0.03932586,
         0.06672903, -0.02758159],
       ...,
       [-0.03216615,  0.01223362, -0.0081108 , ..., -0.02626609,
         0.02875581, -0.00597369],
       [-0.04632537,  0.00109524,  0.00653874, ...,  0.00301363,
         0.01393175, -0.02368219],
       [-0.10451492, -0.04652261, -0.04840812, ..., -0.11269972,
        -0.00916995, -0.02187049]], shape=(30057, 512))

# Check that FWHT is correct
- It should be invertible

In [9]:
len(d_signs)

512

In [10]:
inverted_rot_embeddings = fwht_batch(rot_embeddings, seed=42, sign_flip=False)
np.allclose(embeddings, (inverted_rot_embeddings * d_signs)[:, :384], atol=1e-5)

True

# Load Codebook

In [11]:
with open("../data/codebook/384d_codebook.pkl", "rb") as f:
    codebook = pkl.load(f)

In [12]:
codebook

{'1bits': array([-0.04070159,  0.04088104]),
 '2bits': array([-0.07087235, -0.01866579,  0.02315479,  0.07451284]),
 '3bits': array([-0.09786132, -0.05691854, -0.03069189, -0.00993948,  0.01022873,
         0.03272187,  0.06077609,  0.10216256]),
 '4bits': array([-0.12091945, -0.08460474, -0.06134046, -0.04448182, -0.03125859,
        -0.01997488, -0.00952856,  0.00078499,  0.0112464 ,  0.02168263,
         0.0320071 ,  0.04226933,  0.05339511,  0.06735399,  0.08780586,
         0.12175898])}

In [13]:
len(codebook["4bits"])

16

# Quantize embeddings according to codebook

In [14]:
N_BITS = 4

In [15]:
quantized_emb_buckets = np.argmin(np.abs(rot_embeddings[:, :, np.newaxis] - codebook[f"{int(N_BITS)}bits"].reshape(1, 1, -1)), axis=2).astype(np.uint8)

In [16]:
quantized_emb_buckets

array([[ 6,  5,  9, ...,  7,  7,  6],
       [ 3,  5,  7, ..., 12, 11,  6],
       [ 1,  9,  5, ..., 11, 13,  4],
       ...,
       [ 4,  8,  6, ...,  4, 10,  6],
       [ 3,  7,  8, ...,  7,  8,  5],
       [ 0,  3,  3, ...,  0,  6,  5]], shape=(30057, 512), dtype=uint8)

In [17]:
quantized_emb_buckets[:5, :6,]

array([[ 6,  5,  9,  6,  3,  6],
       [ 3,  5,  7,  5,  7,  8],
       [ 1,  9,  5,  5,  6,  3],
       [10,  3,  7,  6,  3,  8],
       [ 8,  7,  3, 11, 11, 14]], dtype=uint8)

# Since we are using 4 bits, pack every 2 numbers into a single uint8

In [18]:
bit_packed_emb = quantized_emb_buckets[:, 0::2] << 4 | quantized_emb_buckets[:, 1::2]

In [19]:
bit_packed_emb

array([[101, 150,  54, ...,  79, 199, 118],
       [ 53, 117, 120, ...,  57, 204, 182],
       [ 25,  85,  99, ..., 184, 155, 212],
       ...,
       [ 72, 110, 121, ..., 110,  84, 166],
       [ 55, 134,  69, ...,  93, 151, 133],
       [  3,  58,  55, ..., 115, 144, 101]],
      shape=(30057, 256), dtype=uint8)

# Check size of objects

In [20]:
print(f"Object Sizes ({len(df)} samples, {N_BITS}-bit quantization)")
print("-----------------------------------------------------------------")

for name, obj in [("Rotated Embeddings", rot_embeddings), ("Quantized Buckets", quantized_emb_buckets), ("Bit Packed Embeddings", bit_packed_emb)]:
    print(f"{name}: {sys.getsizeof(obj) / (1024 * 1024):.2f} MB")

Object Sizes (30057 samples, 4-bit quantization)
-----------------------------------------------------------------
Rotated Embeddings: 117.41 MB
Quantized Buckets: 14.68 MB
Bit Packed Embeddings: 7.34 MB
